# NewsQA RAG - Phase 1 Retrieval Tournament (Google Colab T4)
Resumable single-GPU tournament. Original questions select winners; resolved questions are supplementary paired analysis.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time
REPO_URL = 'https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT = 'ed6adc30cd44f1e47ad6b60dda8ac7eb71b7476f'
HF_REPO_ID = 'ThomasAnderson2009/newsqa-rag-evaluation'
HF_REVISION = 'v1.0.0'
SMOKE_MODE = True
SMOKE_QUESTIONS = 5
FAST_MODE = SMOKE_MODE
RUN_LATENCY_CALIBRATION = True
LATENCY_REPEATS, LATENCY_QUESTIONS = 3, 100
BACKUP_TO_DRIVE = True
RESTORE_CHECKPOINT = ''
CONTENT = Path('/content')
PROJECT_ROOT = CONTENT / 'Text-Mining---NewsQA-RAG'
WORK_ROOT = CONTENT / ('newsqa_phase1_smoke' if SMOKE_MODE else 'newsqa_phase1')
RESULTS = WORK_ROOT / 'results'
CHECKPOINT_PATH = CONTENT / f'{WORK_ROOT.name}_checkpoint.tar'
DRIVE_OUTPUT = CONTENT / 'drive/MyDrive/newsqa_phase1'
assert REPO_COMMIT != 'SET_TO_COMMIT_CONTAINING_COLAB_RUNNER'


## 1. Secret, Drive, repository and single GPU
In Colab, open the key icon under Secrets, create `HF_TOKEN`, and enable notebook access. Select a T4 GPU runtime before running this cell.

In [ ]:
from google.colab import userdata, drive
token = userdata.get('HF_TOKEN')
assert token, 'Add HF_TOKEN in Colab Secrets and enable notebook access'
os.environ['HF_TOKEN'] = token
os.environ['HF_HOME'] = str(CONTENT / 'hf_cache')
os.environ.update({'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1'})
if BACKUP_TO_DRIVE:
    drive.mount('/content/drive')
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
if not PROJECT_ROOT.exists():
    subprocess.run(['git','clone','--depth','30',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU'
print('GPU:',torch.cuda.get_device_name(0),round(torch.cuda.get_device_properties(0).total_memory/2**30,1),'GiB')
print('Pinned commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_ROOT,text=True).strip())


In [ ]:
import pandas as pd
from IPython.display import display, Image
def backup_outputs():
    if not BACKUP_TO_DRIVE: return
    DRIVE_OUTPUT.mkdir(parents=True,exist_ok=True)
    for source in [CHECKPOINT_PATH, WORK_ROOT/'phase1_results_bundle.zip']:
        if not source.exists():
            print('Backup not available yet:',source,flush=True); continue
        target=DRIVE_OUTPUT/source.name
        print('Copying to Drive:',source,'->',target,flush=True)
        shutil.copy2(source,target)
        print('Backed up:',target,round(target.stat().st_size/2**20,1),'MiB',flush=True)
def run_driver(stage):
    cmd=[sys.executable,'-u','scripts/run_phase1_kaggle.py','--repo-id',HF_REPO_ID,'--revision',HF_REVISION,'--work-root',str(WORK_ROOT),'--stop-after',stage,'--gpu-count','1','--checkpoint-path',str(CHECKPOINT_PATH)]
    if FAST_MODE: cmd.append('--fast')
    if SMOKE_MODE: cmd += ['--smoke-questions',str(SMOKE_QUESTIONS)]
    if RESTORE_CHECKPOINT and stage=='round1' and not WORK_ROOT.exists(): cmd += ['--restore-checkpoint',RESTORE_CHECKPOINT]
    print('$',' '.join(cmd),flush=True)
    try:
        subprocess.run(cmd,cwd=PROJECT_ROOT,check=True,env={**os.environ,'PYTHONUNBUFFERED':'1'})
    except subprocess.CalledProcessError as error:
        print(f'FAILED stage={stage} exit_code={error.returncode}',flush=True)
        raise
    finally:
        backup_outputs()
def show_csv(name,sort='retrieval.mrr@5.mean'):
    frame=pd.read_csv(RESULTS/name)
    if sort in frame: frame=frame.sort_values(sort,ascending=False)
    display(frame); return frame
def paired(frame):
    keys=[k for k in ['index','retriever','reranker','reranker_model','partition'] if k in frame]
    values=[k for k in ['retrieval.mrr@5.mean','retrieval.hit_rate@5.mean','retrieval.ndcg@5.mean'] if k in frame]
    return frame.pivot_table(index=keys,columns='variant',values=values).reset_index()


## 2. Dataset preparation and Round 1
On one GPU, dense model builds are serialized. CPU BM25 construction may run alongside the current lightweight GPU build.

In [ ]:
started=time.time(); run_driver('round1')
round1=show_csv('round1.csv'); display(paired(round1))
print('Minutes:',round((time.time()-started)/60,1)); display(json.loads((RESULTS/'round1_winners.json').read_text()))


## 3. Round 2: retrieval methods and rerankers
All reranker experiments execute serially to avoid repeated large-model memory peaks.

In [ ]:
started=time.time(); run_driver('round2')
round2=show_csv('round2.csv'); display(paired(round2))
print('Minutes:',round((time.time()-started)/60,1)); display(json.loads((RESULTS/'winner_lock.json').read_text()))


## 4. Round 3: chunking ablation
Chunk profiles and their retrieval experiments run sequentially on the single T4.

In [ ]:
started=time.time(); run_driver('round3')
round3=show_csv('round3.csv'); display(paired(round3))
print('Minutes:',round((time.time()-started)/60,1)); display(json.loads((RESULTS/'winner_lock.json').read_text()))


## 5. Locked final-test evaluation

In [ ]:
started=time.time(); run_driver('final')
final_test=show_csv('final_test.csv'); display(paired(final_test))
print('Minutes:',round((time.time()-started)/60,1))


## 6. Serial cache-free latency calibration

In [ ]:
if RUN_LATENCY_CALIBRATION and not FAST_MODE:
    specs=sorted((WORK_ROOT/'specs').glob('round*.yaml'))
    cmd=[sys.executable,'scripts/calibrate_phase1_latency.py',*map(str,specs),'--output',str(RESULTS/'latency_calibration.csv'),'--repeats',str(LATENCY_REPEATS),'--n-eval',str(LATENCY_QUESTIONS)]
    subprocess.run(cmd,cwd=PROJECT_ROOT,check=True,env={**os.environ,'CUDA_VISIBLE_DEVICES':'0'})
    show_csv('latency_calibration.csv',sort='latency.total.p50_ms'); backup_outputs()
else: print('Latency calibration skipped')


## 7. Figures, archives and downloads

In [ ]:
subprocess.run([sys.executable,'-u','scripts/export_phase1_results.py','--experiments-root',str(WORK_ROOT/'experiments'),'--output-dir',str(RESULTS)],cwd=PROJECT_ROOT,check=True,env={**os.environ,'PYTHONUNBUFFERED':'1'})
backup_outputs()
for figure in sorted((RESULTS/'figures').glob('*.png')):
    print(figure.name); display(Image(filename=str(figure)))
for path in sorted(RESULTS.rglob('*')):
    if path.is_file(): print(path.relative_to(RESULTS),round(path.stat().st_size/2**20,2),'MiB')
print('Results archive:',WORK_ROOT/'phase1_results_bundle.zip')
print('Checkpoint:',CHECKPOINT_PATH)
if BACKUP_TO_DRIVE: print('Drive backup:',DRIVE_OUTPUT)
